In [1]:
from ollama import Client
from pydantic import BaseModel
from typing import Literal, Type, List, Dict, Any
import inspect
import json
import webbrowser
import re

def extract_html_from_string(text: str) -> str:
    """
    Extracts and returns the first <html>...</html> block from the input string.
    Discards any other tags like <think></think>.
    Returns the HTML string, or an empty string if not found.
    """
    match = re.search(r'<html.*?</html>', text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0)
    return ""
# Initialize Ollama client with explicit endpoint
client = Client(host="http://localhost:11434")


def text_output(
    model: str = "qwen3:4b",
    history: str = "",
    prompt: str = "",
    system_instructions: str = "Format output as valid html only. Add minimal css to make the html look good.\n\n"
) -> dict:
    """
    Query an Ollama model via Client, enforce a Pydantic schema, and return validated output.

    Parameters:
        model (str): Model identifier (e.g., "qwen3:4b").
        history (str): Previous conversation history.
        prompt (str): The user prompt.

    Returns:
        dict: Parsed and validated model output.

    Raises:
        ValueError: If schema_model is missing or validation fails.
    """

    messages: List[Dict[str, Any]] = []
    if history:
        messages.append({"role": "user", "content": history})
    #messages.append({"role": "system", "content": system_instructions})

    messages.append({"role": "user", "content": prompt + system_instructions})

    response = client.chat(model=model, messages=messages)
    content = response.message.content

    return content

import os
from typing import List, Dict, Any
import pandas as pd
from unstructured.partition.pdf import partition_pdf

def parse_earnings_report(
    file_path: str,
    chunk_size: int = 1000
) -> List[Dict[str, Any]]:
    """
    Parses a company's earnings report PDF into structured chunks suitable for small LLMs.
    
    - Splits document into headings, narrative text, and tables.
    - Chunks long text sections into <chunk_size> characters.
    - Converts tables to CSV strings for easy tokenization.
    
    Args:
        file_path: Path to the PDF earnings report.
        chunk_size: Maximum number of characters per text chunk.
        
    Returns:
        A list of dicts with keys:
            - heading: Section heading under which content resides.
            - type: "text" or "table".
            - content: Text snippet or CSV-formatted table.
    """
    # Partition PDF into elements
    elements = partition_pdf(filename=file_path)
    
    chunks: List[Dict[str, Any]] = []
    current_heading = "Document"
    buffer_text = []
    buffer_tables: List[str] = []

    def flush_buffers():
        nonlocal buffer_text, buffer_tables, current_heading
        text = " ".join(buffer_text).strip()
        # Break long text into smaller chunks
        for i in range(0, len(text), chunk_size):
            snippet = text[i : i + chunk_size]
            chunks.append({
                "heading": current_heading,
                "type": "text",
                "content": snippet
            })
        # Add tables as separate chunks
        for tbl_csv in buffer_tables:
            chunks.append({
                "heading": current_heading,
                "type": "table",
                "content": tbl_csv
            })
        buffer_text = []
        buffer_tables = []

    for elem in elements:
        # Detect headings (Title, Heading)
        if elem.category in ("Title", "Header", "Heading3", "Heading4", "Heading5"):
            # flush previous section
            if buffer_text or buffer_tables:
                flush_buffers()
            current_heading = elem.text.strip()
        elif elem.category in ("NarrativeText", "ListItem"):
            buffer_text.append(elem.text.strip())
        elif elem.category == "Table":
            # Try to convert to DataFrame
            try:
                df = pd.DataFrame(elem.table.rows, columns=elem.table.columns)
                tbl_csv = df.to_csv(index=False)
            except Exception:
                # Fallback to raw text if conversion fails
                tbl_csv = elem.get_text()
            buffer_tables.append(tbl_csv)

    # Flush remaining buffers
    if buffer_text or buffer_tables:
        flush_buffers()

    return chunks

# Example usage:
if __name__ == "__main__":
    path = "path/to/earnings_report.pdf"
    parsed = parse_earnings_report(path, chunk_size=800)
    for chunk in parsed:
        print(f"--- {chunk['heading']} ({chunk['type']}) ---\n{chunk['content'][:200]}...\n")


/home/nitish/Documents/github/PublicReportResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
pytesseract is not installed. Cannot use the ocr_only partitioning strategy. Falling back to partitioning with another strategy.
Falling back to partitioning with hi_res.


PDFPageCountError: Unable to get page count.
I/O Error: Couldn't open file 'path/to/earnings_report.pdf': No such file or directory.


In [2]:
import pdfplumber
import pdfplumber

def pdf_to_text(pdf_path: str) -> str:
    """
    Reads the text content only from a PDF file using pdfplumber and returns the combined text.
    """
    text_content = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                text_content.append(text)
    return "\n".join(text_content)
def pdf_to_html(pdf_path: str) -> str:
    """
    Loads a PDF file using pdfplumber and returns the entire content as an HTML string.
    All tables are formatted as HTML tables, and text is wrapped in <p> tags.
    """
    html_parts = ['<html><body>']
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            html_parts.append(f'<h2>Page {page_num}</h2>')
            # Extract and add text
            text = page.extract_text()
            if text:
                for para in text.split('\n'):
                    html_parts.append(f'<p>{para}</p>')
            # Extract and add tables
            tables = page.extract_tables()
            for table in tables:
                html_parts.append('<table border="1" style="border-collapse:collapse;">')
                for row in table:
                    html_parts.append('<tr>' + ''.join(f'<td>{cell if cell is not None else ""}</td>' for cell in row) + '</tr>')
                html_parts.append('</table>')
    html_parts.append('</body></html>')
    return '\n'.join(html_parts)

file = pdf_to_text("docs/citi_result_2025_q1.pdf")

Cannot set gray non-stroke color because /'P40' is an invalid float value
Cannot set gray non-stroke color because /'P42' is an invalid float value
Cannot set gray non-stroke color because /'P43' is an invalid float value
Cannot set gray non-stroke color because /'P49' is an invalid float value
Cannot set gray non-stroke color because /'P50' is an invalid float value
Cannot set gray non-stroke color because /'P54' is an invalid float value
Cannot set gray non-stroke color because /'P55' is an invalid float value
Cannot set gray non-stroke color because /'P59' is an invalid float value
Cannot set gray non-stroke color because /'P60' is an invalid float value
Cannot set gray non-stroke color because /'P66' is an invalid float value
Cannot set gray non-stroke color because /'P67' is an invalid float value
Cannot set gray non-stroke color because /'P73' is an invalid float value
Cannot set gray non-stroke color because /'P74' is an invalid float value
Cannot set gray non-stroke color becau

In [3]:
print(file[:1000].replace("\n", " "))
file = file.replace("\n", " ")

For Immediate Release Citigroup Inc. (NYSE: C) April 15, 2025 CEO COMMENTARY FIRST QUARTER 2025 RESULTS AND KEY METRICS Citi CEO Jane Fraser said, “With net income of $4.1 billion we delivered a strong quarter, 1Q ROCE marked by continued CET1 momentum, positive operating 1Q 1Q Net 1Q 8.0% Capital leverage and improved returns Revenues Income EPS Ratio in each of our five businesses. 1Q RoTCE Services recorded its best first $21.6B $4.1B $1.96 13.4%(2) quarter revenue in a decade. 9.1%(1) Markets had a good first quarter with revenue up 12% driven by strong client activity and RETURNED ~$2.8 BILLION IN THE FORM OF COMMON DIVIDENDS AND SHARE monetization. Banking was up REPURCHASES 12% with M&A revenue nearly double from what it was last PAYOUT RATIO OF 74%(3) year. Wealth revenues increased 24% with progress across all BOOK VALUE PER SHARE OF $103.90 three client segments. USPB was up 2%, driven mainly by TANGIBLE BOOK VALUE PER SHARE OF $91.52(4) growth in Branded Cards, and also saw 

In [11]:
# Usage:
html = text_output(model='gemma3:12b-it-q8_0', system_instructions="Write the output in html format only. Add minimal CSS to make it look good. \n\n", 
                       prompt=f"""You are an expert financial analyst who looks at earnings reports 
                       of big US banks and interprets them for senior leadership in a finance firm based only on the document provided. 
                       You do not guess. You accurately reflect what is in the attached report.

                       Here is the finance report for Citi enclosed in report tags:

                       <report>{file[:20000]}</report>

                       1. Focus on metrics important to senior leadership of a different finance firm.
                       2. Think about which are the most important metrics and findings from the report
                       3. Extract key financial metrics like revenue, income related metrics, EPS, etc. and other metrics of importance to senior leadership. If some metric is not available, dont include it in tables
                       4. End with key positives for the bank and key negatives for the bank mentioned in the report. Mention the section header as Key Positives for <bank>
                       5. You return the output strictly in html format. Add CSS to make it look good and presentable. Use only steelblue. Choose pleasing and presentation ready colors. Use the appropriate html tags and tables to format the output.

                       Produce a detailed report. Mention the key metrics, key findings, and any actionable insights.
                       """)
html2 = extract_html_from_string(html)
if html2:
    with open("citi.html", "w") as f:
        f.write(html2)
    webbrowser.open("citi.html")

Gtk-Message: 01:25:44.313: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.
Gtk-Message: 01:25:44.399: Failed to load module "canberra-gtk-module"
Gtk-Message: 01:25:44.401: Failed to load module "canberra-gtk-module"
[47569:47569:0630/012544.491745:ERROR:dbus/object_proxy.cc:590] Failed to call method: org.freedesktop.Secret.Service.ReadAlias: object_path= /org/freedesktop/secrets: org.freedesktop.DBus.Error.AccessDenied: An AppArmor policy prevents this sender from sending this message to this recipient; type="method_call", sender=":1.138" (uid=1000 pid=47569 comm="/snap/brave/521/opt/brave.com/brave/brave /home/ni" label="snap.brave.brave (enforce)") interface="org.freedesktop.Secret.Service" member="ReadAlias" error name="(unset)" requested_reply="0" destination="org.freedesktop.secrets" (uid=1000 pid=4439 comm="/usr/bin/gnome-keyring-daemon --foreground --compo" label="unconfined")


### Summarize file - to 5000 Chars and then second LLM summarizes

In [ ]:
import math

summary_size=1000
chunk_size=5000

def after_think_tag(text: str) -> str:
    """
    Returns the part of the string after the first </think> tag.
    If </think> is not found, returns the original string.
    """
    tag = "</think>"
    idx = text.find(tag)
    if idx != -1:
        return text[idx + len(tag):]
    return text

def summarize_chunk(chunk, summary_size, model="qwen3:14b-q8_0"):
    """
    Summarize a chunk of text (~5000 chars) to ~500 chars using the specified LLM.
    """
    prompt = (
        f"Summarize the following financial report content in about {summary_size} characters (roughly {int(int(summary_size)/5)} words), "
        "focusing on the most important financial metrics, findings, and insights. "
        "Be concise and accurate. If there is no important information, keep the summary short. If you are doubtful about some info, leave it out. Do not add information not present in the text.\n\n"
        f"{chunk}"
    )
    return text_output(model=model, prompt=prompt, system_instructions="")

def summarize_long_file(file_text, chunk_size=5000, summary_size=500):
    """
    Splits the file_text into chunks, summarizes each, and returns the concatenated summaries.
    """
    summaries = []
    n_chunks = math.ceil(len(file_text) / chunk_size)
    for i in range(n_chunks):
        chunk = file_text[i*chunk_size : (i+1)*chunk_size]
        summary = after_think_tag(summarize_chunk(chunk, summary_size))
        # Optionally truncate to summary_size
        print(f'chunk: {i}, summary: {summary}')
        summaries.append(summary)
    return "\n".join(summaries)

# Step 1: Summarize each 5000-char chunk to 500 chars using qwen3:4b
summaries = summarize_long_file(file, chunk_size=chunk_size, summary_size=summary_size)

# Step 2: Feed concatenated summaries to qwen3:8b for final HTML report
final_prompt = (
    "You are an expert financial analyst. Based on the following summaries of a bank's earnings report, "
    "produce a detailed, presentation-ready HTML report for senior leadership. "
    "Include key metrics, findings, actionable insights, and format the output as valid HTML with minimal CSS. "
    "Use only steelblue for color accents. Summaries:\n\n"
    f"{summaries}"
)
final_html = text_output(model="qwen3:14b-q8_0", prompt=final_prompt, 
                         system_instructions="Format output as valid HTML only. Add minimal CSS with white background to make the HTML look good.\n\n")

# Extract HTML and save/open
html2 = extract_html_from_string(final_html)
if html2:
    with open("citi_final.html", "w") as f:
        f.write(html2)
    webbrowser.open("citi_final.html")

chunk: 0, summary: 

Citigroup reported Q1 2025 net income of $4.1B ($1.96 EPS) on $21.6B revenue, up 3% YoY. ROCE rose to 13.4%, driven by improved returns across all five businesses. Services hit a decade-high revenue, while Markets grew 12% and Banking rose 12% (M&A revenue doubled). Wealth revenue surged 24%, and USPB grew 2%. The firm returned $2.8B to shareholders via dividends and buybacks, a 74% payout ratio. Operating expenses fell 5% YoY, offset by higher credit costs (+15% to $2.7B). Book value per share was $103.90, tangible book value $91.52.
chunk: 1, summary: 

Citigroup's 1Q25 net income rose 40% YoY to $4.06B, driven by higher revenues ($21.6B) and lower credit losses ($2.46B). End-of-period loans grew 4% to $702B, deposits increased 1% to $1.3T, and book value per share rose 5% to $103.90. CET1 capital ratio fell to 13.4% vs. 13.6% prior, while SLR remained stable at 5.8%. ROCE improved to 8.0% (vs. 5.4% prior), but Services revenue dipped 5% QoQ to $4.89B. Non-accrua

Opening in existing browser session.


[66417:66417:0630/135920.396625:ERROR:ui/gfx/x/atom_cache.cc:232] Add chromium/from-privileged to kAtomsToCache
[66417:66417:0630/140223.270670:ERROR:CONSOLE:1] "Uncaught (in promise) SyntaxError: Unexpected token 'H', "HTTP/1.1 4"... is not valid JSON", source: devtools://devtools/bundled/devtools_app.html?remoteBase=https://devtools.brave.com/serve_file/@253bb9512609910f855ef4f6075ceb87c79b182/&targetType=tab&can_dock=true&panel=elements (1)


In [12]:
# Extract HTML and save/open
html2 = extract_html_from_string(final_html)
if html2:
    with open("citi_final.html", "w") as f:
        f.write(html2)
    webbrowser.open("citi_final.html")

Opening in existing browser session.


In [16]:
import os
from typing import List, Dict, Any
import pandas as pd
from unstructured.partition.pdf import partition_pdf
from ollama import Client

# Parser: split PDF into headings, text chunks, and CSV-formatted tables
from unstructured.partition.pdf import partition_pdf

def parse_earnings_report(
    file_path: str,
    chunk_size: int = 1200
) -> list[dict]:
    """
    Splits a PDF earnings report into structured chunks for LLM input.
    - Headings become section markers.
    - Narrative text is concatenated and chunked at ~chunk_size characters.
    - Tables are converted to CSV strings.
    Does NOT use OCR/tesseract.
    """
    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        pdf_infer_table_structure=True,
        ocr_languages=None,           # disables tesseract OCR
        infer_table_structure=True,   # ensures table structure is inferred
        extract_images_in_pdf=False,  # disables image extraction
        ocr_mode="skip"               # disables OCR fallback
    )

    chunks: list[dict] = []
    current_heading = "Document"
    text_buffer: list[str] = []
    table_buffer: list[str] = []

    def flush_section():
        nonlocal text_buffer, table_buffer
        text = " ".join(text_buffer).strip()
        # chunk text
        for i in range(0, len(text), chunk_size):
            chunks.append({
                "heading": current_heading,
                "type": "text",
                "content": text[i : i + chunk_size]
            })
        # add tables
        for tbl in table_buffer:
            chunks.append({
                "heading": current_heading,
                "type": "table",
                "content": tbl
            })
        text_buffer.clear()
        table_buffer.clear()

    for el in elements:
        if el.category.startswith("Heading"):
            if text_buffer or table_buffer:
                flush_section()
            current_heading = el.text.strip()
        elif el.category in ("NarrativeText", "ListItem"):
            text_buffer.append(el.text.strip())
        elif el.category == "Table":
            try:
                import pandas as pd
                df = pd.DataFrame(el.table.rows, columns=el.table.columns)
                table_buffer.append(df.to_csv(index=False))
            except Exception:
                table_buffer.append(el.get_text())

    # flush remaining
    if text_buffer or table_buffer:
        flush_section()

    return chunks

pdf_file = "docs/citi_result_2025_q1.pdf"
html_insights = extract_insights_as_html_ollama(pdf_file)
# Save to file
with open("insights.html", "w", encoding="utf-8") as f:
    f.write(html_insights)
print("Insights written to insights.html")


Failed to get OCRAgent instance: No module named 'unstructured_pytesseract'


RuntimeError: Could not get the OCRAgent instance. Please check the OCR package and the OCR_AGENT environment variable.